In [9]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchsummary import summary
from PIL import Image

import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn

from load_data import  CatDogDataLoadandSave
from vgg16_model import VGG16

### Load data and Save dataset as ubyte format


In [10]:
data_path = r'/Users/gimoon/Documents/GitHub/Data'
train_data_path = os.path.join(data_path, "cat-and-dog", "training_set")
test_data_path  = os.path.join(data_path, "cat-and-dog", "test_set")



In [11]:
os.path.exists(train_data_path)

True

### create torch data loader

In [28]:
from torch.utils.data import Subset

# Load full dataset
dataset_train = CatDogDataLoadandSave(data_dir=train_data_path)

# Create a subset with reduced number of images
num_train_samples = 1000  # Reduce to 1000 images (adjust as needed)
indices = list(range(min(num_train_samples, len(dataset_train))))
dataset_train_subset = Subset(dataset_train, indices)

print(f"Original dataset size: {len(dataset_train)}")
print(f"Subset dataset size: {len(dataset_train_subset)}")

# Create DataLoader with the subset
train_loader = torch.utils.data.DataLoader(dataset_train_subset, batch_size=64, shuffle=True)

Original dataset size: 8005
Subset dataset size: 1000


In [ ]:
train

### Build VGG16 architecture

In [16]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


model = VGG16(3, 2).to(device)

device

device(type='mps')

In [18]:
from torchsummary import summary

if device.type == 'cpu':
    summary(model, (3, 224, 224))


In [19]:
learning_rate = 1e-4
num_epochs = 1

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [20]:
model.train()

for epoch in range(num_epochs):
    ProgressBar = tqdm(enumerate(train_loader), total=len(train_loader))

    for batch_idx, (inputs, labels) in ProgressBar:

        # Ensure labels are torch.long before moving to device for CrossEntropyLoss
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.long()
        # print(inputs.dtype, labels.dtype)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        #Update Progress bar
        ProgressBar.set_description(f'Epoch [{epoch+1}]')
        ProgressBar.set_postfix(loss=loss.item())

Epoch [1]:   3%|▎         | 4/126 [00:18<09:21,  4.60s/it, loss=0.693]


KeyboardInterrupt: 

In [ ]:
'checkpoint-' + checkpoint['saved_datetime'] + '.pth'

'checkpoint-2026-02-12 10:42:12.pth'